In [1]:
%load_ext autoreload
%autoreload 2

ModuleNotFoundError: No module named 'imp'

In [16]:
%%javascript
require(["https://unpkg.com/higlass-arcs/dist/higlass-arcs.min.js"],
    function(hglib) {
        // Plugin registered
    }
);


<IPython.core.display.Javascript object>

In [2]:
!pip install --pre higlass-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 21.1 MB/s eta 0:00:00


## Synced heatmaps

In [17]:
import higlass as hg

# Configure remote data sources (tilesets)
tileset1 = hg.remote(
    uid="CQMd6V_cRw6iCI_-Unl3PQ",
    server="https://higlass.io/api/v1/",
    name="Rao et al. (2014) GM12878 MboI (allreps) 1kb",
)

tileset2 = hg.remote(
    uid="QvdMEvccQuOxKTEjrVL3wA",
    server="https://higlass.io/api/v1/",
    name="Rao et al. (2014) K562 MboI (allreps) 1kb",
)

# Create a HeatmapTrack for each tileset
track1 = tileset1.track("heatmap")
track2 = tileset2.track("heatmap")

# Create two independent Views, one for each heatmap
view1 = hg.view(track1, width=6)
view2 = hg.view(track2, width=6)

# Lock zoom & location for each View
view_lock = hg.lock(view1, view2)

# Concatenate Views side-by-side, and apply synchronization lock
(view1 | view2).locks(view_lock)

In [4]:
# Lock zoom only for each view
(view1 | view2).locks(zoom=view_lock)

In [5]:
# Lock location only for each view
(view1 | view2).locks(location=view_lock)

In [6]:
# Create additional views and synchronize (apply black to white color range)

bw_color_range = ["rgba(255,255,255,1)", "rgba(0,0,0,1)"]
view3 = hg.view(track1.opts(colorRange=bw_color_range), width=6)
view4 = hg.view(track2.opts(colorRange=bw_color_range), width=6)

# Create stacked view configuration and lock views by column
((view1 / view2) | (view3 / view4)).locks(
    hg.lock(view1, view2),
    hg.lock(view3, view4),
)

## Value scale syncing

In [7]:
# Create Tracks

# creates a hg.Track without a tileset
t1 = hg.track("top-axis")

# Creates a hg.RemoteTileset object
remote_tileset = hg.remote(
    uid="CQMd6V_cRw6iCI_-Unl3PQ",
    server="https://higlass.io/api/v1/",  # optional, "http://higlass.io/api/v1/"
    name="Rao et al. (2014) GM12878 MboI (allreps) 1kb",
)

# Tileset.track() creates a hg.Track object binding the parent Tileset
t2 = remote_tileset.track("heatmap").opts(valueScaleMax=0.5)

# Create Views

# Positional arguments for `hg.view` are overloaded. Keyword arguments are
# forwarded to the creation of the `hg.View`. Layout fields (`x`, `y`,
# `width`, `height`) may also be assigned.

# (1) Track objects (positioning guessed based on track type)
view1 = hg.view(t1, t2, width=6)

# (2) (Track, position) tuple
view2 = hg.view((t1, "top"), t2, width=6)

# (3) hg.Tracks object
view3 = hg.view(hg.Tracks(top=[t1], center=[t2]), width=6)

# (View, Track) tuples -> ScaleValueLock
scale_value_lock = hg.lock((view1, t2), (view2, t2))

(view1 | view2).locks(scale_value_lock)

## Remote heatmaps

In [8]:
# Initialize data sources
tset1 = hg.remote(
    uid="CQMd6V_cRw6iCI_-Unl3PQ",
    name="Rao et al. (2014) GM12878 MboI (allreps) 1kb",
)

tset2 = hg.remote(
    uid="QvdMEvccQuOxKTEjrVL3wA",
    name="Rao et al. (2014) K562 MboI (allreps) 1kb",
)

# Create a track for each data source
t1 = tset1.track("heatmap", height=300)
t2 = tset2.track("heatmap", height=300)

# Create a custom DividedTrack and modify color scale
t3 = hg.divide(t1, t2).opts(
    colorRange=["blue", "white"],
    valueScaleMin=0.1,
    valueScaleMax=10,
)

# Set initial view domains
domain = (7e7, 8e7)
v1 = hg.view(t1, width=4).domain(x=domain)
v2 = hg.view(t2, width=4).domain(x=domain)
v3 = hg.view(t3, width=4).domain(x=domain)

(v1 | v3 | v2).locks(hg.lock(v1, v2, v3))

## Extract track from another view config

In [9]:
# Load view config from URL
url = "https://gist.githubusercontent.com/manzt/c2c498dac3ca9804a2b8e4bc1af3b55b/raw/ee8426c9728e875b6f4d65030c61181c6ba29b53/example.json"
config = hg.Viewconf.from_url(url)

# Display viewconfig
config

In [10]:
# Inspect the Viewconf tracks
for position, track in config.views[0].tracks:
    print(position)  # position in view layout
    print(track)  # python object from `higlass-schema`

left
tilesetUid='OHJakQICQD6gTD7skx4EWA' server='//higlass.io/api/v1' type='vertical-gene-annotations' uid='dqBTMH78Rn6DeSyDBoAEXw' width=60 height=None options={'labelPosition': 'bottomRight', 'name': 'Gene Annotations (hg19)', 'fontSize': 10, 'labelColor': 'black', 'labelBackgroundColor': '#ffffff', 'labelLeftMargin': 0, 'labelRightMargin': 0, 'labelTopMargin': 0, 'labelBottomMargin': 0, 'minHeight': 24, 'plusStrandColor': 'blue', 'minusStrandColor': 'red', 'trackBorderWidth': 0, 'trackBorderColor': 'black', 'showMousePosition': False, 'mousePositionColor': '#000000', 'geneAnnotationHeight': 16, 'geneLabelPosition': 'outside', 'geneStrandSpacing': 4} data=None chromInfoPath=None fromViewUid=None x=None y=None
left
tilesetUid=None server=None type='vertical-chromosome-labels' uid='RHdQK4IRQ7yJeDmKWb7Pcg' width=30 height=None options={'color': '#808080', 'stroke': '#ffffff', 'fontSize': 12, 'fontIsLeftAligned': False, 'showMousePosition': False, 'mousePositionColor': '#000000', 'revers

In [11]:
# Extract a specific track from the above Viewconf and modify its properties
gene_annotation_track = config.views[0].tracks.top[0].properties(height=200)

# Display track in isolation
hg.view(gene_annotation_track)

## Local bigWig files

In [12]:
!wget http://hgdownload.cse.ucsc.edu/goldenpath/hg19/encodeDCC/wgEncodeSydhTfbs/wgEncodeSydhTfbsGm12878InputStdSig.bigWig

--2026-03-08 18:56:25--  http://hgdownload.cse.ucsc.edu/goldenpath/hg19/encodeDCC/wgEncodeSydhTfbs/wgEncodeSydhTfbsGm12878InputStdSig.bigWig
Resolving hgdownload.cse.ucsc.edu (hgdownload.cse.ucsc.edu)... 128.114.119.163
Connecting to hgdownload.cse.ucsc.edu (hgdownload.cse.ucsc.edu)|128.114.119.163|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 302716075 (289M)
Saving to: ‘wgEncodeSydhTfbsGm12878InputStdSig.bigWig’

wgEncodeSydhTfbsGm1 100%[===================>] 288.69M  45.9MB/s    in 7.0s    

2026-03-08 18:56:33 (41.4 MB/s) - ‘wgEncodeSydhTfbsGm12878InputStdSig.bigWig’ saved [302716075/302716075]



In [14]:
tileset = hg.bigwig("wgEncodeSydhTfbsGm12878InputStdSig.bigWig")
hg.view(tileset.track("horizontal-bar"))

ImportError: You must have `clodius` installed to use `hg.bigwig`.

## Local cooler files

This section describes how to load cooler files that are on the same filesystem as the Jupyter notebook.

In [15]:
tileset = hg.cooler("../test/data/Dixon2012-J1-NcoI-R1-filtered.100kb.multires.cool")
hg.view(tileset.track("heatmap"))

ImportError: You must have `clodius` installed to use `hg.cooler`.

<IPython.core.display.Javascript object>

In [23]:
%%javascript
require(["https://unpkg.com/higlass-arcs/dist/higlass-arcs.min.js"], function() {
    console.log("higlass-arcs plugin loaded");
});


import higlass as hg

tileset1 = hg.remote(
    uid="CQMd6V_cRw6iCI_-Unl3PQ",
    server="https://higlass.io/api/v1/",
    name="Rao et al. (2014) GM12878 MboI (allreps) 1kb",
)

tileset2 = hg.remote(
    uid="QvdMEvccQuOxKTEjrVL3wA",
    server="https://higlass.io/api/v1/",
    name="Rao et al. (2014) K562 MboI (allreps) 1kb",
)

# Your arc data tileset (bedpe-style, e.g. from a .beddb file)
arcs_tileset = hg.remote(
    uid="YOUR_ARCS_TILESET_UID",
    server="https://higlass.io/api/v1/",
    name="My Arcs",
)

track1 = tileset1.track("heatmap")
track2 = tileset2.track("heatmap")

# Add arcs track — type "1d-arcs" is registered by higlass-arcs plugin
arcs_track = arcs_tileset.track("1d-arcs")

view1 = hg.view(track1, arcs_track, width=6)
view2 = hg.view(track2, width=6)

view_lock = hg.lock(view1, view2)
(view1 | view2).locks(view_lock)


<IPython.core.display.Javascript object>

In [24]:
view_lock = hg.lock(view1, view2)
(view1 | view2).locks(view_lock)

In [29]:
# Location: chr4:190,998,876-191,000,255
from typing import ClassVar, Literal
import higlass as hg

# Define the plugin track class — this is the correct pattern
class ArcsTrack(hg.PluginTrack):
    type: Literal["1d-arcs"]
    plugin_url: ClassVar[str] = (
        "https://unpkg.com/higlass-arcs/dist/higlass-arcs.min.js"
    )

# Configure remote data sources (tilesets)
tileset1 = hg.remote(
    uid="CQMd6V_cRw6iCI_-Unl3PQ",
    server="https://higlass.io/api/v1/",
    name="Rao et al. (2014) GM12878 MboI (allreps) 1kb",
)

tileset2 = hg.remote(
    uid="QvdMEvccQuOxKTEjrVL3wA",
    server="https://higlass.io/api/v1/",
    name="Rao et al. (2014) K562 MboI (allreps) 1kb",
)

# Create heatmap tracks
track1 = tileset1.track("heatmap")
track2 = tileset2.track("heatmap")

# Create the arcs track using the plugin class directly
arcs_track = ArcsTrack(
    type="1d-arcs",
    tilesetUid="JzccFAJUQEiz-0188xaWZg",
    server="https://resgen.io/api/v1",
    height=80,
    options={
        "labelPosition": "topLeft",
        "strokeColor": "green",
        "strokeWidth": 1.5,
        "strokeOpacity": 0.33,
        "arcStyle": "circle",
        "flip1D": "yes",
        "name": "Neuron PLAC-seq",
    },
)

# Build views — pass arcs_track with explicit position "top"
view1 = hg.view(
    (track1, "center"),
    (arcs_track, "top"),
    width=6,
)
view2 = hg.view(track2, width=6)

# Lock zoom & location
view_lock = hg.lock(view1, view2)

# Render side-by-side
(view1 | view2).locks(view_lock)


AttributeError: 'Viewconf' object has no attribute 'viewconf'

In [32]:
from typing import ClassVar, Literal
import higlass as hg

class ArcsTrack(hg.PluginTrack):
    type: Literal["1d-arcs"]
    plugin_url: ClassVar[str] = (
        "https://unpkg.com/higlass-arcs/dist/higlass-arcs.min.js"
    )

tileset1 = hg.remote(
    uid="CQMd6V_cRw6iCI_-Unl3PQ",
    server="https://higlass.io/api/v1/",
    name="Rao et al. (2014) GM12878 MboI (allreps) 1kb",
)

tileset2 = hg.remote(
    uid="QvdMEvccQuOxKTEjrVL3wA",
    server="https://higlass.io/api/v1/",
    name="Rao et al. (2014) K562 MboI (allreps) 1kb",
)

# ✅ hg19 chromosome sizes tileset — required for the search box
chrom_sizes = hg.remote(
    uid="ADfY_RtsQR6oKOMyrq6qhw",
    server="https://resgen.io/api/v1",
    name="hg19 chrom sizes",
)

track1 = tileset1.track("heatmap")
track2 = tileset2.track("heatmap")

# ✅ Chromosome labels track — this is what the search box anchors to
chrom_track = chrom_sizes.track("horizontal-chromosome-labels")

arcs_track = ArcsTrack(
    type="1d-arcs",
    tilesetUid="JzccFAJUQEiz-0188xaWZg",
    server="https://resgen.io/api/v1",
    height=80,
    options={
        "labelPosition": "topLeft",
        "strokeColor": "green",
        "strokeWidth": 1.5,
        "strokeOpacity": 0.33,
        "arcStyle": "circle",
        "flip1D": "yes",
        "name": "Neuron PLAC-seq",
    },
)

search_box = hg.GenomePositionSearchBox(
    autocompleteServer="//higlass.io/api/v1",
    autocompleteId="OHJakQICQD6gTD7skx4EWA",   # hg19 gene annotations
    chromInfoServer="//higlass.io/api/v1",
    chromInfoId="hg19",
    visible=True,
)

# ✅ Add chrom_track to both views at "top"
view1 = hg.view(
    (track1, "center"),
    (arcs_track, "top"),
    (chrom_track, "top"),       # <-- required
    width=6,
    genomePositionSearchBox=search_box,
)

view2 = hg.view(
    (track2, "center"),
    (chrom_sizes.track("horizontal-chromosome-labels"), "top"),  # <-- required
    width=6,
    genomePositionSearchBox=search_box,
)

view_lock = hg.lock(view1, view2)

vc = (view1 | view2).locks(view_lock)
vc = vc.model_copy(update={
    "trackSourceServers": [
        "//higlass.io/api/v1",
        "https://resgen.io/api/v1",
    ]
})

vc


In [35]:
# GM12878 HiCCUPS loops (hg19) — use https not ftp
!wget "https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSE63525&format=file&file=GSE63525_GM12878_primary%2Breplicate_HiCCUPS_looplist.txt.gz" \
     -O GSE63525_GM12878_loops.txt.gz

# Decompress
!gunzip GSE63525_GM12878_loops.txt.gz

# Preview the format (first 3 lines)
!head -3 GSE63525_GM12878_loops.txt


--2026-03-08 19:15:12--  https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSE63525&format=file&file=GSE63525_GM12878_primary%2Breplicate_HiCCUPS_looplist.txt.gz
Resolving www.ncbi.nlm.nih.gov (www.ncbi.nlm.nih.gov)... 130.14.29.110, 2607:f220:41e:4290::110
Connecting to www.ncbi.nlm.nih.gov (www.ncbi.nlm.nih.gov)|130.14.29.110|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 674624 (659K) [application/octet-stream]
Saving to: ‘GSE63525_GM12878_loops.txt.gz’

GSE63525_GM12878_lo 100%[===================>] 658.81K  --.-KB/s    in 0.07s   

2026-03-08 19:15:12 (9.33 MB/s) - ‘GSE63525_GM12878_loops.txt.gz’ saved [674624/674624]

chr1	x1	x2	chr2	y1	y2	color	o	e_bl	e_donut	e_h	e_v	fdr_bl	fdr_donut	fdr_h	fdr_v	num_collapsed	centroid1	centroid2	radius
10	100225000	100230000	10	100420000	100425000	0,255,255	95	27.4926	25.533	31.2977	32.8853	1.19563917441e-15	9.93567880311e-16	3.82998212108e-10	4.0995892129e-10	10	100228000	100426000	10606.6017178
10	100225000	100230000	10

In [38]:
!clodius aggregate bedpe \
    --assembly hg19 \
    --chr1-col 1 --from1-col 2 --to1-col 3 \
    --chr2-col 4 --from2-col 5 --to2-col 6 \
    --has-header \
    --output-file GM12878_loops.beddb \
    GSE63525_GM12878_loops.txt


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/clodius/cli/aggregate.py", line 233, in line_to_dict
    chrom1_offset = chrom_info.cum_chrom_lengths[chrom1]
                    ~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^
KeyError: '10'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/bin/clodius", line 8, in <module>
    sys.exit(cli())
             ^^^^^
  File "/usr/local/lib/python3.12/dist-packages/click/core.py", line 1485, in __call__
    return self.main(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/click/core.py", line 1406, in main
    rv = self.invoke(ctx)
         ^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/click/core.py", line 1873, in invoke
    return _process_result(sub_ctx.command.invoke(sub_ctx))
                           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/di

In [39]:
# Add "chr" prefix to chromosome columns (skips the header line)
awk 'NR==1{print; next} {$1="chr"$1; $4="chr"$4; print}' OFS='\t' \
    GSE63525_GM12878_loops.txt > GSE63525_GM12878_loops_chr.txt

# Verify it looks right
head -2 GSE63525_GM12878_loops_chr.txt


SyntaxError: invalid syntax (2789273065.py, line 2)